Building BERT (Bidirectional Encoder Representation from Transformers) model requires constructing its core components step by step:

1. BERT Embedding

2. Multi-Head Attention

3. Position-wise Feed-Forward Network:

4. Encoder Layer:

5. BERT Model Assembly

### 1. BERT Embedding Layer

BERT combines three separate embeddings: token IDs, positional indices, and segment Ids (used to distinguish sentence pairs)

In [1]:
import torch
import torch.nn as nn
import math

class BertEmbedding(nn.Module):
    def __init__(self, vocab_size, hidden_size, max_len, token_type_size):
        super().__init__()
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=hidden_size) # Token embedding
        self.positional_embedding = nn.Embedding(
            num_embeddings=max_len, 
            embedding_dim=hidden_size) # Positional embedding
        self.seg_embedding = nn.Embedding(
            num_embeddings=token_type_size, 
            embedding_dim=hidden_size) # Segment embedding

        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, 
                input_ids: torch.Tensor,
                segment_ids: torch.Tensor) -> torch.Tensor:
        if input_ids.shape != segment_ids.shape:
            raise ValueError(
                "input_ids and segment_ids must have the same shape. "
                f"Received {input_ids.shape} and {segment_ids.shape}"
            )

        batch_size, sequence_length = input_ids.shape

        if sequence_length > self.positional_embedding.num_embeddings:
            raise ValueError(
                f"Sequence length {sequence_length} exceeds maximum length "
                f"{self.positional_embedding.num_embeddings}."
            )

        position_ids = torch.arange(
            sequence_length,
            dtype=torch.long,
            device = input_ids.device
        )
        position_ids = position_ids.unsqueeze(0).expand(
            batch_size,
            sequence_length
        )

        token_embedding = self.token_embedding(input_ids)
        positional_embedding= self.positional_embedding(position_ids)
        seg_embedding =  self.seg_embedding(segment_ids)
        # sum all three representations together
        embedding = (
            token_embedding
            + positional_embedding 
            + seg_embedding
        )
        return self.dropout(self.norm(embedding))

vocab_size = 10
hidden_size = 5
max_len = 20
token_type_size = 2

bertEmbedding = BertEmbedding(vocab_size, hidden_size, max_len, token_type_size)

input_ids = torch.tensor([
    [1,2,3,4],
    [1,2,3,4]
])
segment_ids = torch.tensor([
    [0,0,0,0],
    [1,1,1,1]
])

output = bertEmbedding(input_ids, segment_ids)

print(f"input ids: {input_ids.shape}")
print(f"output: {output.shape}")
print(output)

input ids: torch.Size([2, 4])
output: torch.Size([2, 4, 5])
tensor([[[ 1.3713, -0.1646,  0.5821,  0.1842, -1.9729],
         [-0.1388, -1.2867,  1.3099,  1.2358, -1.1202],
         [-0.5190,  0.0987,  1.3690,  0.8617, -1.8103],
         [ 1.2467, -1.8439, -0.2449,  1.0552, -0.2131]],

        [[ 2.0844,  0.2065, -0.9112, -0.6468, -0.7328],
         [ 0.8773, -1.9162,  0.3772,  0.0000, -0.4968],
         [ 0.0000,  0.0000,  0.0000,  0.4112, -2.1607],
         [ 1.5875, -1.1377, -0.0000,  0.1663,  0.7265]]],
       grad_fn=<MulBackward0>)


### 2. Multi-Head Attention Mechanism

The multi-head attention component splits the input vectors into multiple 'heads' to let the model look at the sequence from different representation perspective simultaneously. 
This is similar to stack of kernels in convolution operation.

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.num_heads = num_heads # num_heads
        self.hidden_size = hidden_size # hidden_dim
        self.head_dim = hidden_size // num_heads

        self.q_linear = nn.Linear(hidden_size, hidden_size) # Q = qWq + bq
        self.k_linear = nn.Linear(hidden_size, hidden_size) # K = kWk + bk
        self.v_linear = nn.Linear(hidden_size, hidden_size) # V = vWv + bv
        self.out_linear = nn.Linear(hidden_size, hidden_size)

    # Although their input values are initially the same, they pass through different learned layers:
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0) # batch

        # Project Linearly; After this projection - Q != K != V
        Q = self.q_linear(q) # batch x seq_length (query_length) x hidden_dim        
        K = self.k_linear(k) # batch x seq_length (key_length)   x hidden_dim
        V = self.v_linear(v) # batch x seq_length (value_length) x hidden_dim
        
        # Split into heads: (batch, seq_length, num_heads, head_dim)
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim) # batch x seq_length x num_heads x head_dim
        K = K.view(batch_size, -1, self.num_heads, self.head_dim) # batch x seq_length x num_heads x head_dim
        V = V.view(batch_size, -1, self.num_heads, self.head_dim) # batch x seq_length x num_heads x head_dim

        # Move the head dimension: (batch, num_heads, seq_length, head_dim)
        Q = Q.transpose(1,2) # batch x num_heads x seq_length x head_dim
        K = K.transpose(1,2) # batch x num_heads x seq_length x head_dim
        V = V.transpose(1,2) # batch x num_heads x seq_length x head_dim

        '''
        Batch item
            ├── Head 0
            │   ├── token 1 query
            │   ├── token 2 query
            │   └── ...
            ├── Head 1
            │   ├── token 1 query
            │   ├── token 2 query
            │   └── ...
            └── Head 2
                ├── token 1 query
                ├── token 2 query
                └── ...
        '''
        # Scaled dot-Product Attention 
        # Self Attention: batch x heads x seq_length (query_length) x seq_length (key_length)
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.head_dim) # (Q @ K)/sqrt(d_k)

        if mask is not None:
            # Mask out padding tokens (e.g. set to highly negative value)
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V) # batch, heads, seq_length, head_dim

        # Concatenate heads back together: batch x seq_length x hidden_size
        context = context.transpose(1,2).contiguous().view(batch_size, -1, self.hidden_size)

        return self.out_linear(context)


#### Test with Tensors

In [4]:
# Example 
import torch
import torch.nn as nn

batch_size = 2
sequence_length = 5
hidden_size = 12
num_heads = 3
head_dim = hidden_size / num_heads

hidden_states = torch.randn(
    batch_size,
    sequence_length,
    hidden_size,
)

q_linear = nn.Linear(hidden_size, hidden_size)
k_linear = nn.Linear(hidden_size, hidden_size)
v_linear = nn.Linear(hidden_size, hidden_size)

q = q_linear(hidden_states)
k = k_linear(hidden_states)
v = v_linear(hidden_states)

print(q.shape)
print(q[0, :, :2])
print(k.shape)
print(k[0, :, :2])
print(v.shape)
print(v[0, :, :2])

torch.Size([2, 5, 12])
tensor([[ 0.2736,  0.2289],
        [-0.6066, -0.3568],
        [-0.8925, -0.0177],
        [-0.4464, -0.4989],
        [ 0.8786,  0.8174]], grad_fn=<SliceBackward0>)
torch.Size([2, 5, 12])
tensor([[ 0.4113,  0.2153],
        [-0.1958, -0.6290],
        [ 0.3675, -0.3402],
        [ 0.2235, -0.1608],
        [ 0.0979,  0.1482]], grad_fn=<SliceBackward0>)
torch.Size([2, 5, 12])
tensor([[-0.4804, -0.1115],
        [-0.0283, -0.1409],
        [-0.1631, -0.5587],
        [-0.2724, -0.3027],
        [-0.1741,  0.0514]], grad_fn=<SliceBackward0>)


In [5]:
multihead = MultiHeadAttention(hidden_size, num_heads)
multihead_output = multihead(hidden_states, hidden_states, hidden_states)

multihead_output.shape

torch.Size([2, 5, 12])

### 3. Position-Wise Feed-Forward Network

This is an MLP applied to each token position individually. BERT natively uses the GELU activation function.

The purpose is to transform each token representation independently after self-attention has mixed information between tokens.

In [3]:
class FeedForward(nn.Module):
    def __init__(self, hidden_size, intermediate_size, drop_out=0.1):
        super().__init__()
        # Expanding the representation gives the model more capacity to learn combinations of hidden features.
        # expansion ratio: intermediate_size = 4 × hidden_size
        self.linear1 = nn.Linear(hidden_size, intermediate_size) # h = xW₁ + b₁
        self.activation = nn.GELU() # z = a(h)
        self.linear2 = nn.Linear(intermediate_size, hidden_size) # h = zW₁ + b₁
        self.drop_out = nn.Dropout(drop_out)

    def forward(self, x):
        # FFN(x) = W₂ GELU(W₁x + b₁) + b₂
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return self.drop_out(x) #

### 4. Encoder Layer

This combines Multi-Head Attention and the Feed Forward network with residual connections and layer normalization.

In [10]:
class EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size, drop_out):
        super().__init__()
        self.attention = MultiHeadAttention(hidden_size, num_heads) # Multihead self attention
        self.norm1 = nn.LayerNorm(hidden_size)
        self.ffn = FeedForward(hidden_size, intermediate_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.drop_out = nn.Dropout(drop_out)

    def forward(self, x, mask=None):
        # Attention sub-layer with residual connection
        attn_out = self.attention(x, x, x, mask)
        x = self.norm1(x + self.drop_out(attn_out))

        # Feed forward sub-layer with residual connection
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

#### 5. Assemble the full BERT Model

Here we dynamically layer multiple encoder blocks and provide a simple linear pooler for downstream classification tasks (which processes the [CLS] token hidden state)

In [11]:
class BERT(nn.Module):
    def __init__(self, 
                 vocab_size, hidden_size=768,
                 num_layers=12, num_heads=12,
                 max_len=512, token_type_size=2,
                 intermediate_size=3072, drop_out=0.1):
        super().__init__()

        if hidden_size % num_heads != 0:
            raise ValueError(
                "hidden_size must be divisible by num_heads. "
                f"Received hidden_size={hidden_size}, "
                f"num_heads={num_heads}."
            )
        
        self.embeddings = BertEmbedding(vocab_size, hidden_size, max_len, token_type_size)
        self.encoder_blocks = nn.ModuleList([
            EncoderLayer(hidden_size, num_heads, intermediate_size, drop_out) for _ in range(num_layers)
        ])
        self.pooler = nn.Linear(
            hidden_size, 
            hidden_size
            )
        self.pooler_activation = nn.Tanh()
    
    def forward(self, input_ids, segment_ids, mask=None):
        # 1. Embed inputs
        x = self.embeddings(input_ids, segment_ids)

        # 2. Reshape attention mask for multi-head attention structure
        # (batch, 1, 1, seq_len)
        if mask is not None:
            if mask.shape != input_ids.shape:
                raise ValueError(
                    "mask and input_ids must have the same shape. "
                    f"Received {mask.shape} and {input_ids.shape}."
                )

            # (batch, sequence) -> (batch, 1, 1, sequence)
            mask = mask[:, None, None, :]

        # 3. Pass through sequential encoder layers
        for layer in self.encoder_blocks:
            x = layer(x, mask)

        # 4. Extract [CLS] token representation (index 0) for classification pooling
        cls_hidden = x[:, 0, :]
        pooled_output = self.pooler_activation(self.pooler(cls_hidden))

        return x, pooled_output # Final token states and pooled [CLS] state


from dataclasses import dataclass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@dataclass
class BERTConfig:
    """ Configuration for BERT model """
    vocab_size: int = 30522
    num_layers: int = 12
    hidden_size: int = 768
    num_heads: int = 12
    max_len: int = 512
    token_type_size: int = 2
    intermediate_size: int = 3072
    drop_out: float = 0.1

bert_config = BERTConfig()

bert_model = BERT(
    bert_config.vocab_size,
    bert_config.hidden_size,
    bert_config.num_layers,
    bert_config.num_heads,
    bert_config.max_len,
    bert_config.token_type_size,
    bert_config.intermediate_size,
    bert_config.drop_out
).to(device)

bert_model

BERT(
  (embeddings): BertEmbedding(
    (token_embedding): Embedding(30522, 768)
    (positional_embedding): Embedding(512, 768)
    (seg_embedding): Embedding(2, 768)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder_blocks): ModuleList(
    (0-11): 12 x EncoderLayer(
      (attention): MultiHeadAttention(
        (q_linear): Linear(in_features=768, out_features=768, bias=True)
        (k_linear): Linear(in_features=768, out_features=768, bias=True)
        (v_linear): Linear(in_features=768, out_features=768, bias=True)
        (out_linear): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (activation): GELU(approximate='none')
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (drop_out): Drop

### Verification

In [59]:
vocab_size = 30000
batch_size = 2
seq_length = 10

dummy_model = BERT(
    vocab_size=vocab_size,
    num_layers=4,
    num_heads=4,
    hidden_size=256,
    intermediate_size=1024,
)

dummy_input = torch.randint(0, vocab_size, (batch_size, seq_length))
dummy_segment_ids = torch.zeros((batch_size, seq_length), dtype=torch.long)
dummy_mask = torch.ones(batch_size, seq_length) # 1 for valid tokens; 0 for pads

# Run inference
sequence_output, pooled_output = model(dummy_input, dummy_segment_ids, dummy_mask)

print("Sequence Hidden States Shape:", sequence_output.shape) # Expected: [2, 10, 256]
print("Pooled [CLS] Output Shape:", pooled_output.shape)       # Expected: [2, 256]

Sequence Hidden States Shape: torch.Size([2, 10, 256])
Pooled [CLS] Output Shape: torch.Size([2, 256])


In [15]:
class MaskedLanguageModelHead(nn.Module):
    """Predicts original tokens from masked states."""
    def __init__(self, embed_dim, vocab_size):
        super().__init__()
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.activation = nn.GELU()
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, hidden_states):
        x = self.norm(self.activation(self.linear(hidden_states)))
        return self.decoder(x)

class NextSentencePredictionHead(nn.Module):
    """Binary classifier evaluating if Sentence B naturally follows Sentence A."""
    def __init__(self, embed_dim):
        super().__init__()
        self.linear = nn.Linear(embed_dim, 2) # Two outputs: Is Next (1) or Not Next (0)
        
    def forward(self, pooler_output):
        return self.linear(pooler_output)

class BERTLM(nn.Module):
    """Full BERT framework containing core Transformer and joint pre-training tasks."""
    def __init__(self, bert_core, vocab_size, embed_dim=768):
        super().__init__()
        self.bert = bert_core
        self.mlm_head = MaskedLanguageModelHead(embed_dim, vocab_size)
        self.nsp_head = NextSentencePredictionHead(embed_dim)
        
    def forward(self, input_ids, segment_ids, attention_mask):
        # Propagate through our original BERT core stack
        hidden_states, pooler_output = self.bert(input_ids, segment_ids, attention_mask)
        
        # Calculate individual head classifications
        mlm_scores = self.mlm_head(hidden_states)
        nsp_scores = self.nsp_head(pooler_output)
        
        return mlm_scores, nsp_scores

In [19]:
from bert_dataset import *

# if __name__ == "__main__":
# 1. Toy document sample dataset
sample_corpus = [
    "Welcome to learning machine learning from scratch",
    "Deep learning models require plenty of matrix multiplication operations",
    "PyTorch provides native tensor support with dynamic computational graph architecture",
    "Transformers utilize self attention blocks to model long sequence data dependencies",
    "Natural language processing is evolving rapidly with larger language models"
]

# Initialize components
tokenizer = SimpleTokenizer(sample_corpus)
dataset = BERTDataset(sample_corpus, tokenizer, max_len=16)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)

# 2. Build the Model Core and Pre-training Container
# Using smaller dimensions for structural unit testing
bert_base_config = BERT(
    vocab_size=tokenizer.vocab_size, 
    hidden_size=64, 
    num_layers=2, 
    num_heads=2, 
    intermediate_size=128, 
    max_len=16,
    token_type_size=2,
    drop_out=0.1
)
bert_pretrain_model = BERTLM(bert_base_config, vocab_size=tokenizer.vocab_size, embed_dim=64)

# 3. Define Loss Objectives
criterion_mlm = nn.CrossEntropyLoss(ignore_index=-100) # Skips unmasked text computations
criterion_nsp = nn.CrossEntropyLoss()

# 4. Process One Batch
sample_batch = next(iter(data_loader))

mlm_output, nsp_output = bert_pretrain_model(
    input_ids=sample_batch["input_ids"],
    segment_ids=sample_batch["segment_ids"],
    attention_mask=sample_batch["attention_mask"]
)

# 5. Compute joint multi-task loss
loss_mlm = criterion_mlm(mlm_output.view(-1, tokenizer.vocab_size), sample_batch["mlm_labels"].view(-1))
loss_nsp = criterion_nsp(nsp_output, sample_batch["nsp_label"])
total_loss = loss_mlm + loss_nsp

print("--- Execution Check Successful ---")
print(f"Vocabulary Size: {tokenizer.vocab_size} tokens")
print(f"MLM Target Matrix Shape: {mlm_output.shape} -> (Batch, Seq, Vocab)")
print(f"NSP Target Matrix Shape: {nsp_output.shape} -> (Batch, Classes)")
print(f"Computed Masked Language Model Loss Value: {loss_mlm.item():.4f}")
print(f"Computed Next Sentence Prediction Loss Value: {loss_nsp.item():.4f}")
print(f"Total Consolidated Batch Loss: {total_loss.item():.4f}")

--- Execution Check Successful ---
Vocabulary Size: 46 tokens
MLM Target Matrix Shape: torch.Size([4, 16, 46]) -> (Batch, Seq, Vocab)
NSP Target Matrix Shape: torch.Size([4, 2]) -> (Batch, Classes)
Computed Masked Language Model Loss Value: 4.1261
Computed Next Sentence Prediction Loss Value: 0.6651
Total Consolidated Batch Loss: 4.7912
